In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기

True

In [2]:
# 모델 설정
model = init_chat_model("openai:gpt-5.6-luna")

# 벡터 저장소 설정
# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 저장된 벡터 DB 가져와야
load_vs = Chroma(
    collection_name="k_ladder_2026",
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

In [4]:
retreiver_mmr = load_vs.as_retriever(search_type="mmr", 
                                    search_kwargs={"k": 5, 
                                                "fetch_k": 50,
                                                "lambda_mult":0.25})

# 실제로 검색



In [5]:
SYSTME_PROMPT = """
너는 공공 정책 안내 도우미다.
아래 자료를 참고해서 답해라.
참고 자료에 없으면 "자료에 없음" 이라고 말해라
정확한 자격, 금액, 기한은 공고 확인이 필요하다고 꼭 덧붙여라
답 끝에 참고한 페이지 번호를 [p.60] 형식으로 적어라
"""

# 검색 결과 문서를 받았을때 메타데이터와 내용을 합쳐서 text 로 변환하는 함수 작성
def format_docs(docs):
    context = ""

    for doc in docs:
        context += f" [p.{doc.metadata["page"]}] \n {doc.page_content} \n\n"

    return context      

In [ ]:
chain = retreiver_mmr | format_docs # 검색한 결과를 form
chain 

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001E4612B8470>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 50, 'lambda_mult': 0.25})
| RunnableLambda(format_docs)

In [7]:
chain.invoke("청년 월세 지원 정책 찾아줘")

' [p.39] \n 037모두의 정책 K-희망사다리 2026\n여성청소년 \n생리용품 지원\n지원대상 \t • \t기초생활수급(생계·의료·주거·교육급여),\t법정차상위계층,\t한부모가족\t지원\t\n대상\t가구의\t9~24세\t여성청소년\n핵심내용 \t •\t 여성청소년\t생리용품\t바우처\t지원(월\t1만\t4,000원),\t국민행복카드로\t구매\n •9세가\t되는\t해의\t1월\t1일부터\t24세가\t끝나는\t해의\t12월\t31일까지\t지원\t\n이용방법 \t •\t 온라인\t신청:\t복지로(www.bokjiro.go.kr)\t또는\t모바일\t앱\n\t •방문\t신청:\t읍·면·동\t주민센터\t및\t행정복지센터\t\n  ※  지원대상 결정 전·후 청소년 본인 또는 신청인(바우처 신청서상의 신청인) 명의의 국민\n행복카드를 발급받아야 사용 가능 \n문의처\t• 성평등가족부\t청소년정책과(☎02-2100-6242)\n\t •한국사회보장정보원(☎1566-3232)\n\t •읍·면·동\t주민센터\t및\t행정복지센터\t\t\n02-2100-6242\n성평등가족부 청소년정책과\n1만  4,000원\n월\t지원금 \n\n [p.14] \n 청년미래적금\n 1600-5500\n금융위원회\n012따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도\n지원대상 \t •\t 일정\t소득\t이하\t만\t19~34세\t청년(병역\t최대\t6년\t인정)\n\t \t - \t\t일반형:\t개인\t소득\t6,000만\t원\t이하\t소득자\t또는\t연\t매출\t3억\t원\t이하\t소상공인\t\n중\t가구\t중위소득\t200%\t이하\n\t \t - \t\t우대형:\t개인소득\t3,600만\t원\t이하\t중소기업\t재직자\t또는\t연\t매출\t1억\t원\t\n이하\t소상공인\t중\t가구\t중위소득\t150%\t이하\n   ※  일반형 요건을 충족하는 중소기업 신규 재직자는 우대형 분류\n핵심내용 \t • \t만기\t3년\n\t •\t납입액(월\t50만\t원\t한도)에\t대

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# rag_prompt 
rag_prompt = ChatPromptTemplate.from_messages({
    ('system', SYSTME_PROMPT),
    ('human','참고자료\n{context} \n질문{question}')
})


rag_chain = {"context": (retreiver_mmr | format_docs),
             "question" : RunnablePassthrough() } | rag_prompt  | model | StrOutputParser()

rag_chain.invoke("청년 월세 지원 정책 찾아줘.")

'참고자료에는 **청년 월세 지원의 구체적인 지원 대상·금액·신청 기간·신청 방법**이 나와 있지 않습니다. **자료에 없음**.\n\n다만 ‘혜택알리미’에 가입하고 개인정보 활용에 동의하면, 청년 월세 지원을 포함해 개인의 소득·재산·거주 상황에 맞는 정부 혜택을 맞춤 추천받고 신청까지 연계할 수 있습니다.\n\n- 대상: 대한민국 모든 국민\n- 이용 방법: 혜택알리미 가입 후 맞춤 안내 이용\n- 문의: 행정안전부 국민맞춤서비스과 ☎ 044-205-2806\n\n청년 월세 지원의 **정확한 자격, 지원 금액, 신청 기한은 최신 공고를 반드시 확인해야 합니다.**\n\n[p.256]'

In [13]:
from pydantic import BaseModel, Field

class AnswerStyle(BaseModel):
    answer : str = Field(description="최종 답변")
    source : str = Field(description="출처")

structured_model = model.with_structured_output(AnswerStyle, method="json_schema")


rag_chain = {"context": (retreiver_mmr | format_docs),
             "question" : RunnablePassthrough() } | rag_prompt  | structured_model 

result = rag_chain.invoke("청년 월세 지원 정책 찾아줘.")

In [14]:
result.model_dump()

{'answer': '청년 월세 지원 정책의 구체적인 지원 대상·금액·신청 기간·신청 방법은 참고자료에 없습니다. 다만 ‘혜택알리미’에서 개인의 상황과 소득·재산 등을 분석해 청년월세 지원 대상에 해당할 가능성이 있으면 맞춤 안내하고, 신청까지 연계할 수 있습니다. 혜택알리미는 대한민국 모든 국민이 이용할 수 있습니다. 정확한 자격, 지원 금액, 신청 기한은 해당 연도 공고를 반드시 확인해야 합니다. [p.256]',
 'source': 'K-희망사다리 2026, p.256'}

In [ ]:
rag_chain = {"context": (retreiver_mmr | format_docs),
             "question" : RunnablePassthrough() } | rag_prompt  | structured_model 

result = rag_chain.invoke("청년 월세 지원 정책 찾아줘.")